# 01 Data Analysis And Prep
Этапы 1-5: загрузка источника, базовая диагностика, подготовка и сохранение промежуточного датасета на диск.

In [8]:
import pandas as pd

from _shared_notebook_utils import DATA_DIR, ROOT, ensure_dirs, save_json, save_pickle

ensure_dirs()
print('ROOT =', ROOT)
print('DATA_DIR =', DATA_DIR)

ROOT = C:\Users\Dmitry\code-projects\diploma-crop-rotation
DATA_DIR = C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data


## 1) Загрузка исходных данных
Проверяем только каталог `dataset`: сначала `*.pkl`, затем директории `*.gdb`. Если источник в `dataset` не найден, пробуем загрузить из cache; если cache отсутствует, выбрасываем ошибку.

In [9]:
dataset_dir = ROOT / 'dataset'
cache_path = ROOT / 'national1724.pkl'

def _load_from_dataset_folder(base_dir):
    if not base_dir.exists() or not base_dir.is_dir():
        return None, None

    pkl_candidates = sorted(base_dir.glob('*.pkl'))
    if pkl_candidates:
        src = pkl_candidates[0]
        return pd.read_pickle(src), src

    gdb_candidates = sorted([p for p in base_dir.iterdir() if p.is_dir() and p.suffix.lower() == '.gdb'])
    if gdb_candidates:
        gdb_path = gdb_candidates[0]
        try:
            import geopandas as gpd
        except ImportError as exc:
            raise ImportError(
                'Найден .gdb в dataset, но geopandas не установлен. Установите geopandas или положите .pkl в dataset.'
            ) from exc

        gdf = gpd.read_file(gdb_path)
        if gdf is None or len(gdf) == 0:
            raise ValueError(f'Не удалось прочитать данные из .gdb: {gdb_path}')
        return gdf, gdb_path

    return None, None

df_raw, source_path = _load_from_dataset_folder(dataset_dir)
source_kind = 'dataset'

if df_raw is None:
    if cache_path.exists():
        df_raw = pd.read_pickle(cache_path)
        source_path = cache_path
        source_kind = 'cache'
    else:
        dataset_status = f"exists={dataset_dir.exists()}"
        raise FileNotFoundError(
            'Источник не найден. Ожидался файл *.pkl или директория *.gdb внутри каталога dataset, '
            'либо cache файл national1724.pkl в корне проекта. '
            f'dataset: {dataset_dir} ({dataset_status}); cache: {cache_path}'
        )

# Type guard for downstream cells.
if df_raw is None:
    raise RuntimeError('df_raw must be loaded before diagnostics.')
df_raw = pd.DataFrame(df_raw).copy()

print('source_kind =', source_kind)
print('source_path =', source_path)
print('df_raw shape:', df_raw.shape)
display(df_raw.head(3))

source_kind = dataset
source_path = C:\Users\Dmitry\code-projects\diploma-crop-rotation\dataset\national1724.pkl
df_raw shape: (16418258, 20)


,CSBID,CSBYEARS,CSBACRES,CDL2017,CDL2018,CDL2019,CDL2020,CDL2021,CDL2022,CDL2023,CDL2024,STATEFIPS,STATEASD,ASD,CNTY,CNTYFIPS,INSIDE_X,INSIDE_Y,Shape_Length,Shape_Area
0,351724000000001,1724,2.777993,24,24,152,152,152,152,152,152,35,3530,30,Union,059,-650336.7078,1.447441e+06,628.315309,11242.149046
1,351724000000002,1724,9.293377,24,24,24,24,24,24,24,152,35,3530,30,Union,059,-648203.5992,1.447142e+06,1042.818836,37608.996229
2,351724000000003,1724,5.775892,24,24,24,24,24,24,24,152,35,3530,30,Union,059,-647571.4914,1.447064e+06,1164.537470,23374.226788


## 2) Первичная диагностика
Фиксируем список CDL-колонок и базовые показатели качества данных.

In [10]:
cdl_cols = [c for c in df_raw.columns if c.startswith('CDL') and c[3:7].isdigit()]
id_cols = [c for c in ['CSBID', 'CSBACRES', 'STATEFIPS', 'CNTYFIPS', 'INSIDE_X', 'INSIDE_Y'] if c in df_raw.columns]

diag_df = pd.DataFrame({
    'column': id_cols + cdl_cols[:10],
    'dtype': [str(df_raw[c].dtype) for c in id_cols + cdl_cols[:10]],
    'missing': [int(df_raw[c].isna().sum()) for c in id_cols + cdl_cols[:10]]
})

print('CDL columns found:', len(cdl_cols))
display(diag_df)

CDL columns found: 8


,column,dtype,missing
0,CSBID,str,0
1,CSBACRES,float64,0
2,STATEFIPS,str,0
3,CNTYFIPS,str,0
4,INSIDE_X,float64,0
5,INSIDE_Y,float64,0
6,CDL2017,int32,0
7,CDL2018,int32,0
8,CDL2019,int32,0
9,CDL2020,int32,0


## 3) Подготовка и очистка
Перенесен рабочий prep-блок из legacy scripts: маппинг CDL code -> name/group, добавление `CDL*_name`/`CDL*_group`, unknown-диагностика и фильтрация строк с bad/unknown классами.

In [11]:
if not isinstance(df_raw, pd.DataFrame):
    raise TypeError('df_raw должен быть pandas DataFrame после шага загрузки')

CDL_YEARS = list(range(2017, 2025))
DEFAULT_DROP_COLUMNS = ["Shape_Length", "Shape_Area", "CSBYEARS"]

CDL_AGGREGATION = {
    1: "corn", 2: "cotton", 3: "rice", 4: "sorghum", 5: "soybeans", 6: "sunflower",
    10: "peanuts", 11: "specialty_crops", 12: "specialty_crops", 13: "specialty_crops", 14: "specialty_crops",
    21: "other_cereals", 22: "wheat", 23: "wheat", 24: "wheat", 25: "other_cereals", 26: "double_crop", 27: "other_cereals", 28: "other_cereals", 29: "other_cereals", 30: "other_cereals",
    31: "oilseeds_other", 32: "oilseeds_other", 33: "oilseeds_other", 34: "oilseeds_other", 35: "oilseeds_other",
    36: "forage_hay", 37: "forage_hay", 38: "oilseeds_other", 39: "specialty_crops",
    41: "sugar_crops", 42: "legumes", 43: "root_tubers", 44: "specialty_crops", 45: "sugar_crops", 46: "root_tubers",
    47: "vegetables_melons", 48: "vegetables_melons", 49: "vegetables_melons", 50: "vegetables_melons",
    51: "legumes", 52: "legumes", 53: "legumes", 54: "vegetables_melons", 55: "fruits_berries",
    56: "specialty_crops", 57: "specialty_crops", 58: "forage_hay", 59: "forage_hay", 60: "forage_hay",
    61: "fallow", 62: "pasture_grass", 63: "non_ag_natural", 64: "non_ag_natural", 65: "non_ag_natural",
    66: "orchards_vineyards_tree_crops", 67: "orchards_vineyards_tree_crops", 68: "orchards_vineyards_tree_crops",
    69: "orchards_vineyards_tree_crops", 70: "orchards_vineyards_tree_crops", 71: "orchards_vineyards_tree_crops",
    72: "orchards_vineyards_tree_crops", 74: "orchards_vineyards_tree_crops", 75: "orchards_vineyards_tree_crops",
    76: "orchards_vineyards_tree_crops", 77: "orchards_vineyards_tree_crops",
    81: "developed_water_wetlands", 82: "developed_water_wetlands", 83: "developed_water_wetlands",
    87: "developed_water_wetlands", 88: "developed_water_wetlands", 92: "developed_water_wetlands",
    111: "developed_water_wetlands", 112: "developed_water_wetlands", 121: "developed_water_wetlands", 122: "developed_water_wetlands",
    123: "developed_water_wetlands", 124: "developed_water_wetlands", 131: "non_ag_natural", 141: "non_ag_natural",
    142: "non_ag_natural", 143: "non_ag_natural", 152: "non_ag_natural", 176: "pasture_grass",
    190: "developed_water_wetlands", 195: "developed_water_wetlands",
    204: "orchards_vineyards_tree_crops", 205: "other_cereals", 206: "vegetables_melons", 207: "vegetables_melons",
    208: "root_tubers", 209: "vegetables_melons", 210: "orchards_vineyards_tree_crops", 211: "orchards_vineyards_tree_crops",
    212: "orchards_vineyards_tree_crops", 213: "vegetables_melons", 214: "vegetables_melons",
    215: "orchards_vineyards_tree_crops", 216: "vegetables_melons", 217: "orchards_vineyards_tree_crops",
    218: "orchards_vineyards_tree_crops", 219: "vegetables_melons", 220: "orchards_vineyards_tree_crops",
    221: "fruits_berries", 222: "vegetables_melons", 223: "orchards_vineyards_tree_crops", 224: "legumes",
    225: "double_crop", 226: "double_crop", 227: "vegetables_melons", 228: "double_crop", 229: "vegetables_melons",
    230: "double_crop", 231: "double_crop", 232: "double_crop", 233: "double_crop", 234: "double_crop",
    235: "double_crop", 236: "double_crop", 237: "double_crop", 238: "double_crop", 239: "double_crop",
    240: "double_crop", 241: "double_crop", 242: "fruits_berries", 243: "vegetables_melons", 244: "vegetables_melons",
    245: "vegetables_melons", 246: "root_tubers", 247: "root_tubers", 248: "vegetables_melons",
    249: "vegetables_melons", 250: "fruits_berries", 254: "double_crop",
}

CDL_CODE_TO_NAME = {
    1: "Corn", 2: "Cotton", 3: "Rice", 4: "Sorghum", 5: "Soybeans", 6: "Sunflower",
    10: "Peanuts", 11: "Tobacco", 12: "Sweet Corn", 13: "Pop or Orn Corn", 14: "Mint",
    21: "Barley", 22: "Durum Wheat", 23: "Spring Wheat", 24: "Winter Wheat", 25: "Other Small Grains",
    26: "Dbl Crop WinWht/Soybeans", 27: "Rye", 28: "Oats", 29: "Millet", 30: "Speltz",
    31: "Canola", 32: "Flaxseed", 33: "Safflower", 34: "Rape Seed", 35: "Mustard",
    36: "Alfalfa", 37: "Other Hay/Non Alfalfa", 38: "Camelina", 39: "Buckwheat",
    41: "Sugarbeets", 42: "Dry Beans", 43: "Potatoes", 44: "Other Crops", 45: "Sugarcane",
    46: "Sweet Potatoes", 47: "Misc Vegs & Fruits", 48: "Watermelons", 49: "Onions", 50: "Cucumbers",
    51: "Chick Peas", 52: "Lentils", 53: "Peas", 54: "Tomatoes", 55: "Caneberries",
    56: "Hops", 57: "Herbs", 58: "Clover/Wildflowers", 59: "Sod/Grass Seed", 60: "Switchgrass",
    61: "Fallow/Idle Cropland", 62: "Pasture/Grass", 63: "Forest", 64: "Shrubland", 65: "Barren",
    66: "Cherries", 67: "Peaches", 68: "Apples", 69: "Grapes", 70: "Christmas Trees",
    71: "Other Tree Crops", 72: "Citrus", 74: "Pecans", 75: "Almonds", 76: "Walnuts", 77: "Pears",
    81: "Clouds/No Data", 82: "Developed", 83: "Water", 87: "Wetlands", 88: "Nonag/Undefined", 92: "Aquaculture",
    111: "Open Water", 112: "Perennial Ice/Snow", 121: "Developed/Open Space", 122: "Developed/Low Intensity",
    123: "Developed/Med Intensity", 124: "Developed/High Intensity", 131: "Barren",
    141: "Deciduous Forest", 142: "Evergreen Forest", 143: "Mixed Forest", 152: "Shrubland",
    176: "Grassland/Pasture", 190: "Woody Wetlands", 195: "Herbaceous Wetlands",
    204: "Pistachios", 205: "Triticale", 206: "Carrots", 207: "Asparagus", 208: "Garlic",
    209: "Cantaloupes", 210: "Prunes", 211: "Olives", 212: "Oranges", 213: "Honeydew Melons",
    214: "Broccoli", 215: "Avocados", 216: "Peppers", 217: "Pomegranates", 218: "Nectarines",
    219: "Greens", 220: "Plums", 221: "Strawberries", 222: "Squash", 223: "Apricots", 224: "Vetch",
    225: "Dbl Crop WinWht/Corn", 226: "Dbl Crop Oats/Corn", 227: "Lettuce", 228: "Dbl Crop Triticale/Corn",
    229: "Pumpkins", 230: "Dbl Crop Lettuce/Durum Wht", 231: "Dbl Crop Lettuce/Cantaloupe",
    232: "Dbl Crop Lettuce/Cotton", 233: "Dbl Crop Lettuce/Barley", 234: "Dbl Crop Durum Wht/Sorghum",
    235: "Dbl Crop Barley/Sorghum", 236: "Dbl Crop WinWht/Sorghum", 237: "Dbl Crop Barley/Corn",
    238: "Dbl Crop WinWht/Cotton", 239: "Dbl Crop Soybeans/Cotton", 240: "Dbl Crop Soybeans/Oats",
    241: "Dbl Crop Corn/Soybeans", 242: "Blueberries", 243: "Cabbage", 244: "Cauliflower",
    245: "Celery", 246: "Radishes", 247: "Turnips", 248: "Eggplants", 249: "Gourds",
    250: "Cranberries", 254: "Dbl Crop Barley/Soybeans",
}

DROP_CLASSES = {
    "double_crop",
    "developed_water_wetlands",
    "non_ag_natural",
    "pasture_grass",
    "orchards_vineyards_tree_crops",
}

def get_cdl_columns(df: pd.DataFrame, years=CDL_YEARS):
    expected = [f"CDL{year}" for year in years]
    return [col for col in expected if col in df.columns], [col for col in expected if col not in df.columns]

def add_cdl_name_columns(df: pd.DataFrame, cdl_columns, code_to_name: dict, unknown_label: str = "Unknown"):
    result = df.copy()
    unknown_counts = {}
    for code_col in cdl_columns:
        mapped = result[code_col].map(code_to_name)
        result[f"{code_col}_name"] = mapped.fillna(unknown_label)
        unknown_counts[f"{code_col}_name"] = int(mapped.isna().sum())
    return result, unknown_counts

def add_cdl_group_columns(df: pd.DataFrame, cdl_columns, code_to_group: dict, unknown_label: str = "other_unknown"):
    result = df.copy()
    unknown_counts = {}
    for code_col in cdl_columns:
        mapped = result[code_col].map(code_to_group)
        result[f"{code_col}_group"] = mapped.fillna(unknown_label)
        unknown_counts[f"{code_col}_group"] = int(mapped.isna().sum())
    return result, unknown_counts

def drop_unneeded_columns(df: pd.DataFrame, columns_to_drop=DEFAULT_DROP_COLUMNS):
    existing = [col for col in columns_to_drop if col in df.columns]
    return df.drop(columns=existing).copy(), existing

def build_unknown_code_report(df_prepared: pd.DataFrame, cdl_columns, unknown_name_label: str = "Unknown", unknown_group_label: str = "other_unknown"):
    rows = []
    for code_col in cdl_columns:
        name_col = f"{code_col}_name"
        group_col = f"{code_col}_group"
        if name_col not in df_prepared.columns or group_col not in df_prepared.columns:
            continue
        mask_unknown = (df_prepared[name_col] == unknown_name_label) | (df_prepared[group_col] == unknown_group_label)
        if not mask_unknown.any():
            continue
        counts = df_prepared.loc[mask_unknown, code_col].value_counts(dropna=False)
        year = int(code_col.replace("CDL", ""))
        for code_value, count in counts.items():
            rows.append({"year": year, "cdl_column": code_col, "code": code_value, "count": int(count)})
    unknown_by_year = pd.DataFrame(rows)
    if unknown_by_year.empty:
        return (
            pd.DataFrame(columns=["year", "cdl_column", "code", "count"]),
            pd.DataFrame(columns=["code", "count"]),
        )
    unknown_by_year = unknown_by_year.sort_values(["count", "year", "code"], ascending=[False, True, True]).reset_index(drop=True)
    unknown_by_code = (
        unknown_by_year.groupby("code", dropna=False)["count"]
        .sum().to_frame(name="count").reset_index()
        .sort_values(by="count", ascending=False).reset_index(drop=True)
    )
    return unknown_by_year, unknown_by_code

def print_preparation_diagnostics(diagnostics: dict):
    print("CDL columns found:", diagnostics["cdl_columns"])
    print("Missing CDL columns:", diagnostics["missing_cdl_columns"])
    print("Dropped technical columns:", diagnostics["dropped_columns"])
    print("Total unknown names:", sum(diagnostics["unknown_name_counts"].values()))
    print("Total unknown groups:", sum(diagnostics["unknown_group_counts"].values()))

cdl_columns, missing_columns = get_cdl_columns(df_raw, years=CDL_YEARS)
if not cdl_columns:
    raise ValueError("No CDL columns found for the specified years.")

df_prepared, unknown_name_counts = add_cdl_name_columns(
    df_raw, cdl_columns=cdl_columns, code_to_name=CDL_CODE_TO_NAME, unknown_label="Unknown"
)
df_prepared, unknown_group_counts = add_cdl_group_columns(
    df_prepared, cdl_columns=cdl_columns, code_to_group=CDL_AGGREGATION, unknown_label="other_unknown"
)
df_prepared, dropped_columns = drop_unneeded_columns(df_prepared, columns_to_drop=DEFAULT_DROP_COLUMNS)

prep_diagnostics = {
    "cdl_columns": cdl_columns,
    "missing_cdl_columns": missing_columns,
    "added_name_columns": [f"{col}_name" for col in cdl_columns],
    "added_group_columns": [f"{col}_group" for col in cdl_columns],
    "dropped_columns": dropped_columns,
    "unknown_name_counts": unknown_name_counts,
    "unknown_group_counts": unknown_group_counts,
}
print_preparation_diagnostics(prep_diagnostics)

unknown_by_year, unknown_by_code = build_unknown_code_report(
    df_prepared=df_prepared,
    cdl_columns=prep_diagnostics["cdl_columns"],
    unknown_name_label="Unknown",
    unknown_group_label="other_unknown",
)

group_columns = [col for col in prep_diagnostics["added_group_columns"] if col in df_prepared.columns]
name_columns = [col for col in prep_diagnostics["added_name_columns"] if col in df_prepared.columns]
bad_group_classes = set(DROP_CLASSES) | {"other_unknown"}
mask_bad_group = df_prepared[group_columns].isin(bad_group_classes).any(axis=1) if group_columns else pd.Series(False, index=df_prepared.index, dtype=bool)
mask_unknown_name = (df_prepared[name_columns] == "Unknown").any(axis=1) if name_columns else pd.Series(False, index=df_prepared.index, dtype=bool)
mask_drop = mask_bad_group | mask_unknown_name

rows_before_cleanup = len(df_prepared)
df_prepared_filtered = df_prepared.loc[~mask_drop].copy()
rows_after_cleanup = len(df_prepared_filtered)

prep_source = "scripts_migrated_prepare_and_cleanup"
print("Rows before filtering:", rows_before_cleanup)
print("Rows removed:", int(mask_drop.sum()))
print("Rows after filtering:", rows_after_cleanup)
print("Unknown rows total:", int(unknown_by_year["count"].sum()) if not unknown_by_year.empty else 0)
print("Prepared shape:", df_prepared.shape)
print("Prepared filtered shape:", df_prepared_filtered.shape)

CDL columns found: ['CDL2017', 'CDL2018', 'CDL2019', 'CDL2020', 'CDL2021', 'CDL2022', 'CDL2023', 'CDL2024']
Missing CDL columns: []
Dropped technical columns: ['Shape_Length', 'Shape_Area', 'CSBYEARS']
Total unknown names: 115
Total unknown groups: 115
Rows before filtering: 16418258
Rows removed: 8307388
Rows after filtering: 8110870
Unknown rows total: 115
Prepared shape: (16418258, 33)
Prepared filtered shape: (8110870, 33)


## 4) Сохранение артефактов этапа
Сохраняем DataFrame и metadata для downstream ноутбуков.

In [12]:
prepared_path = DATA_DIR / '01_df_prepared_filtered.pkl'
size_mb = save_pickle(df_prepared_filtered, prepared_path)

meta = {
    'source_cache': str(cache_path),
    'prep_source': prep_source,
    'prepared_path': str(prepared_path),
    'rows': int(df_prepared_filtered.shape[0]),
    'cols': int(df_prepared_filtered.shape[1]),
    'cdl_cols_count': int(len(cdl_cols))
}

meta_path = DATA_DIR / '01_meta.json'
save_json(meta, meta_path)
print('Saved:', prepared_path, f'({size_mb:.2f} MB)')
print('Saved:', meta_path)

Saved: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data\01_df_prepared_filtered.pkl (2786.60 MB)
Saved: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data\01_meta.json
